In [5]:
# Import Necessary Library

import BAC0
import time
import os
import csv
import datetime
import sys

# Disable
def blockPrint():
    sys.stdout = open(os.devnull, 'w')

# Restore
def enablePrint():
    sys.stdout = sys.__stdout__

# Return BACnet Device

def get_bac0_device(device, network):
    print("Get BAC0 device")
    name, vendor, address, device_id = device
    return BAC0.device(address, device_id, network, poll=0, object_list=None)

# Main Function to Create CSV file and Log the Data

def main():
    try:
        # BAC0 Setup
        bacnet = BAC0.connect()
        bacnet.discover()

        if not bacnet.devices:
            print("No BACnet devices found. Exiting.")
            return

        # Use the first device from the list of discovered devices
        device = bacnet.devices[0]
        # Fetch All info of BACnet Device like Device Name, Device Vendor Details, Device Address, Device ID
        bac0_device = get_bac0_device(device, bacnet)

        # Points to log
        points_to_log = [
            "Temperature.Indoor",
            "Temperature.Water",
            "Temperature.Outdoor",
            "SetPoint.Value",
            "Setpoint.1",
            "Setpoint.2",
            "Setpoint.3",
            "State.Heater",
            "State.Chiller",
            "State",
            "State.VentilationLevel"
        ]

        # Get the current date in the format dd-mm-yyyy
        current_date = datetime.datetime.now().strftime('%d-%m-%Y')
        csv_filename = f"{current_date}.csv"

        # Check if the CSV file already exists
        if not os.path.isfile(csv_filename):
            # If the file doesn't exist, write the header
            with open(csv_filename, 'w', newline='') as csv_file:
                csv_writer = csv.writer(csv_file)
                csv_writer.writerow(['TimeStamp'] + points_to_log)

        # Main Loop of Reading and Logging TAG data
        while True:
            # Get list of points (Tags) with their latest values
            points = bac0_device.points
            
            # Ignor this code.
            blockPrint()
            print(points)
            enablePrint()
            
            # Prepare data to be logged (transposed for horizontal format)
            data_to_log = [datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')] + [point.lastValue for point in points if point.properties.name in points_to_log]
            #print("Logging data:", data_to_log)
            
            # Check if the date has changed
            new_date = datetime.datetime.now().strftime('%d-%m-%Y')
            if new_date != current_date:
                # If the date has changed, close the current CSV file and create a new one
                current_date = new_date
                csv_filename = f"{current_date}.csv"
                with open(csv_filename, 'w', newline='') as csv_file:
                    csv_writer = csv.writer(csv_file)
                    csv_writer.writerow(['TimeStamp'] + points_to_log)

            # Log data into the CSV file
            with open(csv_filename, 'a', newline='') as csv_file:
                csv_writer = csv.writer(csv_file)
                csv_writer.writerow(data_to_log)

            # Wait for 4 seconds before sending updated data
            time.sleep(4)

    except KeyboardInterrupt:
        print("KeyboardInterrupt: Exiting gracefully.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Run Main Function

if __name__ == "__main__":
    # Run Main Function
    main()

2023-12-16 20:47:56,989 - INFO    | Starting BAC0 version 23.07.03 (Lite)
2023-12-16 20:47:57,004 - INFO    | Use BAC0.log_level to adjust verbosity of the app.
2023-12-16 20:47:57,006 - INFO    | Ex. BAC0.log_level('silence') or BAC0.log_level('error')
2023-12-16 20:47:57,008 - INFO    | Starting TaskManager
2023-12-16 20:47:57,035 - WARNING | Netifaces not installed on your system. BAC0 can't detect the subnet.
Please provide subnet for now, we'll consider 255.255.255.0 (/24).
You can install netifaces using 'pip install netifaces'.
2023-12-16 20:47:57,036 - INFO    | Using ip : 192.168.1.7 on port 47808 | broadcast : 192.168.1.255
2023-12-16 20:47:57,154 - INFO    | Starting app...
2023-12-16 20:47:57,157 - INFO    | BAC0 started
2023-12-16 20:47:57,159 - INFO    | Registered as Simple BACnet/IP App
2023-12-16 20:47:57,164 - INFO    | Device instance (id) : 3056456
2023-12-16 20:47:57,168 - INFO    | Update Local COV Task started (required to support COV)
2023-12-16 20:48:01,176 - I